# Create Time Series of Temperature and Electricity Demand for a Given Event


In [ ]:
# Start by importing the packages we need:
import os
import datetime

import pandas as pd
import matplotlib.pyplot as plt

from datetime import timedelta


## Set the Directory Structure

In [ ]:
# Identify the data input and output directories:
metadata_input_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/'
hw_cs_data_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/thermal_events_data/'
temp_data_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/temperature_data/'
load_data_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/load_data/'
image_output_dir =  '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/figures/case_study_results/'


## Write a Function to Process the Temperature Time Series Data


In [ ]:
# Define a function to process the time series of temperature for a given NERC TPL-08 region:
def process_temperature_time_series(temp_data_dir: str, region: str):
    
    # Read in the raw time series data for all NERC regions:
    temp_df = pd.read_csv((temp_data_dir + 'NERC_Region_Daily_Temperature_1980_to_2024.csv'))
    
    # Subset to just the data for NERC region you want to use:
    subset_df = temp_df[(temp_df['Region'] == region)].copy()

    ## Set 'Date' to a datetime variable:
    subset_df['Date'] = pd.to_datetime(subset_df['Date'])
    
    # Add the day of year to be used as an averaging parameter:
    subset_df['DoY'] = subset_df['Date'].dt.dayofyear

    # Calculate the mean T_Min and T_Max by day of year:
    subset_df['T_Min_Mean'] = subset_df.groupby('DoY')['T_Min'].transform('mean').round(2)
    subset_df['T_Max_Mean'] = subset_df.groupby('DoY')['T_Max'].transform('mean').round(2)

    # Calculate the temperature anomalies:
    subset_df['T_Min_Delta'] = subset_df['T_Min'] - subset_df['T_Min_Mean']
    subset_df['T_Max_Delta'] = subset_df['T_Max'] - subset_df['T_Max_Mean']
    
    # Only keep the columns we need:
    output_df = subset_df[['Date','T_Min','T_Min_Mean','T_Min_Delta','T_Max','T_Max_Mean','T_Max_Delta']].copy()
    
    return output_df


In [ ]:
# Test the function:
temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, 
                                          region = 'CA')

temp_df


## Write a Function to Process the Load Time Series Data


In [ ]:
def process_load_time_series(load_data_dir: str, region: str):

    # Read in the load data and subset to a given year:
    load_df = pd.read_csv((load_data_dir + 'WECC_Hourly_Loads_1980_to_2025.csv'))
       
    # Only keep the columns we need:
    subset_df = load_df[['Time_UTC', 'WECC_Load_MWh', (region + '_Load_MWh')]].copy()
    subset_df.rename(columns={'WECC_Load_MWh': 'Interconnection_Load', (region + '_Load_MWh'): 'Region_Load'}, inplace=True)
   
    # Set 'Time_UTC' to a datetime variable:
    subset_df['Time_UTC'] = pd.to_datetime(subset_df['Time_UTC'])

    # Add the hour of year to be used as an averaging parameter:
    subset_df['HoY'] = (((subset_df['Time_UTC'].dt.dayofyear -1) * 24) + subset_df['Time_UTC'].dt.hour)
    
    # Calculate the mean load by hour of year:
    subset_df['Interconnection_Load_Mean'] = subset_df.groupby('HoY')['Interconnection_Load'].transform('mean').round(2)
    subset_df['Region_Load_Mean'] = subset_df.groupby('HoY')['Region_Load'].transform('mean').round(2)

    # Calculate the load anomalies:
    subset_df['Interconnection_Load_Delta'] = subset_df['Interconnection_Load'] - subset_df['Interconnection_Load_Mean']
    subset_df['Region_Load_Delta'] = subset_df['Region_Load'] - subset_df['Region_Load_Mean']

    # Only keep the columns we need:
    output_df = subset_df[['Time_UTC','Interconnection_Load','Interconnection_Load_Mean','Interconnection_Load_Delta','Region_Load','Region_Load_Mean','Region_Load_Delta']].copy()
    
    return output_df
    

In [ ]:
# Test the function:
load_df = process_load_time_series(load_data_dir = load_data_dir, 
                                   region = 'CA')

load_df


## Read in the Heat Wave or Cold Snap Library


In [ ]:
# Extract the heat wave or cold snap library data for a given NERC region:
hw_cs_df = pd.read_csv((hw_cs_data_dir + 'hw_library_expanded.csv'))

hw_cs_df


## Make the Plot


In [ ]:
def plot_event_time_series(event: str, window: int, hw_cs_data_dir: str, temp_data_dir: str, load_data_dir: str, metadata_input_dir: str,
                           image_output_dir: str, image_resolution: int, save_images=False):

    # Read in the heat wave event library:
    hw_cs_df = pd.read_csv((hw_cs_data_dir + 'hw_library_expanded.csv'))
    
    # Subset to just the event specified:
    hw_cs_df = hw_cs_df.loc[hw_cs_df['UID'] == event].copy()

    # Extract the event parameters used to query the functions defined above:
    region = hw_cs_df.loc[hw_cs_df['UID'] == event, 'Region'].item()
    start_date = pd.to_datetime(hw_cs_df.loc[hw_cs_df['UID'] == event, 'Start'].item()) - pd.Timedelta(1, "d")
    end_date = pd.to_datetime(hw_cs_df.loc[hw_cs_df['UID'] == event, 'End'].item()) + pd.Timedelta(1, "d")
    
    # Read in NERC region name file and extract the name:
    nerc = pd.read_csv((metadata_input_dir + 'nerc_tpl08_region_names.csv'))
    nerc_name = nerc.loc[nerc['short_name'] == region, 'long_name'].item()
        
    # Process the temperature time series and subset the data to just dates within the time window:
    temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, region = region)
    temp_df['Date'] = pd.to_datetime(temp_df['Date'])
    temp_subset_df = temp_df[(temp_df['Date'] >= pd.to_datetime((start_date - pd.Timedelta(days=window)))) & (temp_df['Date'] <= pd.to_datetime((end_date + pd.Timedelta(days=window))))].copy()
    peak_temp_subset_df = temp_df[(temp_df['Date'] >= pd.to_datetime((start_date))) & (temp_df['Date'] <= pd.to_datetime((end_date)))].copy()

    # Process the load time series and subset the data to just dates within the time window:
    load_df = process_load_time_series(load_data_dir = load_data_dir, region = region)
    load_df['Time_UTC'] = pd.to_datetime(load_df['Time_UTC'])
    load_subset_df = load_df[(load_df['Time_UTC'] >= pd.to_datetime(start_date - pd.Timedelta(days=window))) & (load_df['Time_UTC'] <= pd.to_datetime(end_date + pd.Timedelta(days=window)))].copy()
    peak_load_subset_df = load_df[(load_df['Time_UTC'] >= pd.to_datetime(start_date)) & (load_df['Time_UTC'] <= pd.to_datetime(end_date))].copy()

    # Calculate the minimum and maximum values to be used in plotting:
    temp_min = min([temp_subset_df['T_Max'].min(),temp_subset_df['T_Max_Mean'].min()]) - 3
    temp_max = max([temp_subset_df['T_Max'].max(),temp_subset_df['T_Max_Mean'].max()]) + 3
    region_load_min = min([load_subset_df['Region_Load'].min(),load_subset_df['Region_Load_Mean'].min()]) * 0.9
    region_load_max = max([load_subset_df['Region_Load'].max(),load_subset_df['Region_Load_Mean'].max()]) * 1.1
    wecc_load_min = min([load_subset_df['Interconnection_Load'].min(),load_subset_df['Interconnection_Load_Mean'].min()]) * 0.9
    wecc_load_max = max([load_subset_df['Interconnection_Load'].max(),load_subset_df['Interconnection_Load_Mean'].max()]) * 1.1
    
    # Calculate the temperature and load anomalies:
    temp_peak = peak_temp_subset_df['T_Max'].max().round(1)
    temp_anom = peak_temp_subset_df.loc[peak_temp_subset_df['T_Max'].idxmax(), ['T_Max_Delta']].item().round(1)
    region_load_peak = peak_load_subset_df['Region_Load'].max().round(0)
    region_load_anom = peak_load_subset_df.loc[peak_load_subset_df['Region_Load'].idxmax(), ['Region_Load_Delta']].item().round(0)
    wecc_load_peak = peak_load_subset_df['Interconnection_Load'].max().round(0)
    wecc_load_anom = peak_load_subset_df.loc[peak_load_subset_df['Interconnection_Load'].idxmax(), ['Interconnection_Load_Delta']].item().round(0)

    # Adjust the start and end dates by 12 hours if they're the same:
    if start_date == end_date:
       event_start_date = start_date - pd.Timedelta(0.5, "d")
       event_end_date = end_date + pd.Timedelta(0.5, "d")
    else:
       event_start_date = start_date
       event_end_date = end_date

    # Make the plot:
    plt.figure(figsize=(25,30))
    plt.rcParams['font.size'] = 18
    plt.rcParams['axes.axisbelow'] = True
    
    ax1 = plt.subplot(311)
    plt.plot(temp_subset_df['Date'], temp_subset_df['T_Max'], color='r', linestyle='-', linewidth=3)
    plt.plot(temp_subset_df['Date'], temp_subset_df['T_Max_Mean'], color='k', linestyle='--', linewidth=2)
    plt.fill_between([pd.to_datetime(event_start_date), pd.to_datetime(event_end_date)], temp_min, temp_max, color='r', alpha=0.15)
    plt.text(0.8, 0.925, ('Hottest Temperature = ' + str(temp_peak) + '$^\circ$F'), fontsize=21, horizontalalignment='center', verticalalignment='center', transform=ax1.transAxes)
    plt.text(0.8, 0.825, ('Temperature Anomaly = ' + str(temp_anom) + '$^\circ$F'), fontsize=21, horizontalalignment='center', verticalalignment='center', transform=ax1.transAxes)
    plt.xlim([pd.to_datetime(start_date - pd.Timedelta(days=window)), pd.to_datetime(end_date + pd.Timedelta(days=window))])
    plt.ylim([temp_min, temp_max])
    plt.ylabel('Max Temp. [$^\circ$F]', fontsize=18)
    plt.title(('Daily Maximum Temperature in ' + nerc_name))

    ax2 = plt.subplot(312)
    plt.plot(load_subset_df['Time_UTC'], load_subset_df['Region_Load'], color='m', linestyle='-', linewidth=3)
    plt.plot(load_subset_df['Time_UTC'], load_subset_df['Region_Load_Mean'], color='k', linestyle='--', linewidth=2)
    plt.fill_between([pd.to_datetime(event_start_date), pd.to_datetime(event_end_date)], region_load_min, region_load_max, color='r', alpha=0.15)
    plt.text(0.8, 0.925, ('Peak Demand = ' + str(region_load_peak) + ' MWh'), fontsize=21, horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes)
    plt.text(0.8, 0.825, ('Peak Demand Anomaly = ' + str(region_load_anom) + ' MWh'), fontsize=21, horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes)
    plt.xlim([pd.to_datetime(start_date - pd.Timedelta(days=window)), pd.to_datetime(end_date + pd.Timedelta(days=window))])
    plt.ylim([region_load_min, region_load_max])
    plt.ylabel('Regional Demand [MWh]', fontsize=18)
    plt.title(('Hourly Electricity Demand in ' + nerc_name))

    ax3 = plt.subplot(313)
    plt.plot(load_subset_df['Time_UTC'], load_subset_df['Interconnection_Load'], color='m', linestyle='-', linewidth=3)
    plt.plot(load_subset_df['Time_UTC'], load_subset_df['Interconnection_Load_Mean'], color='k', linestyle='--', linewidth=2)
    plt.fill_between([pd.to_datetime(event_start_date), pd.to_datetime(event_end_date)], wecc_load_min, wecc_load_max, color='r', alpha=0.15)
    plt.text(0.8, 0.925, ('Peak Demand = ' + str(wecc_load_peak) + ' MWh'), fontsize=21, horizontalalignment='center', verticalalignment='center', transform=ax3.transAxes)
    plt.text(0.8, 0.825, ('Peak Demand Anomaly = ' + str(wecc_load_anom) + ' MWh'), fontsize=21, horizontalalignment='center', verticalalignment='center', transform=ax3.transAxes)
    plt.xlim([pd.to_datetime(start_date - pd.Timedelta(days=window)), pd.to_datetime(end_date + pd.Timedelta(days=window))])
    plt.ylim([wecc_load_min, wecc_load_max])
    plt.ylabel('Interconnection Demand [MWh]', fontsize=18)
    plt.title(('Hourly Electricity Demand in the Western Interconnection'))
    
    # If the "save_images" flag is set to true then save the plot to a .png file:
    if save_images == True:
       if event == 'HW_NERC4_Event60':
          filename = (os.path.join(image_output_dir + 'ED1-GB_Time_Series.png'))
       if event == 'HW_NERC4_Event9':
          filename = (os.path.join(image_output_dir + 'ED2-GB_Time_Series.png'))
       if event == 'HW_NERC1_Event79':
          filename = (os.path.join(image_output_dir + 'HC1-CA_Time_Series.png'))
       if event == 'HW_NERC1_Event8':
          filename = (os.path.join(image_output_dir + 'HC2-CA_Time_Series.png'))  
       if event == 'HW_NERC11_Event62':
          filename = (os.path.join(image_output_dir + 'SE1-PNW_Time_Series.png'))
       if event == 'HW_NERC11_Event88':
          filename = (os.path.join(image_output_dir + 'SE2-PNW_Time_Series.png'))
       plt.savefig(filename, dpi=image_resolution, bbox_inches='tight')
       #plt.close()

    return load_subset_df


In [ ]:
# Test the function:
output_df = plot_event_time_series(event = 'HW_NERC4_Event60',
                                   window = 5, # Days before and after the start of the event to make the plot
                                   hw_cs_data_dir = hw_cs_data_dir,
                                   temp_data_dir = temp_data_dir, 
                                   load_data_dir = load_data_dir,
                                   metadata_input_dir = metadata_input_dir,
                                   image_output_dir = image_output_dir, 
                                   image_resolution = 150, 
                                   save_images = True)

output_df
